In [1]:
import os
import re
import pandas as pd
import logging
from underthesea import chunk
from sklearn.model_selection import train_test_split
import ast
import time
from gensim import corpora, models

In [2]:
QUERY_DATASET_PATH = "dataset\\restaurant_dataset_ver1.csv"
DATASET_PATH = "dataset\\foody_combined_data_final.csv"
QUERY_TEXT_PATH = "dataset\\restaurant_queries.txt"
base_path = os.getcwd()

# To be change if needed
actual_query_dataset_path = os.path.join(base_path, "..", "..", QUERY_DATASET_PATH)
actual_dataset_path = os.path.join(base_path, "..", "..", DATASET_PATH)
actual_query_text_path = os.path.join(base_path, "..", "..", QUERY_TEXT_PATH)

In [3]:
stopwords = []
with open("stopwords.txt", 'r', encoding='utf-8') as f:
    stopwords = f.read().split()


In [4]:
time_related_words = ["hôm nay", "hôm qua", "ngày mai", "tuần này", "tuần sau", "tháng này", "tháng sau", "năm nay", "năm sau"] 
time_of_day_related_words = ["sáng", "trưa", "chiều", "tối", "đêm"]
time_of_week_related_words = ["thứ 2", "thứ 3", "thứ 4", "thứ 5", "thứ 6", "thứ 7", "chủ nhật", "thứ hai", "thứ ba", "thứ tư", "thứ năm", "thứ sáu", "thứ bảy"]
time_amount_related_words = ["h", 'giờ', 'phút', 'p']

location_related_words = ["quận", "huyện", "phường", "xã", "thị trấn", "tỉnh", "thành phố", "đường", "phố", "ngõ", "hẻm"]
good_attribute_words = ["tốt", "ngon", "đẹp", "rẻ", "vui vẻ", "thân thiện", "sạch sẽ", "đáng tiền", "hợp lý"]
invert_attribute_words = ["không", "chẳng", "chả", "chưa", "chẳng phải", "không phải"]
amount_related_words = ["nhiều", "ít", "đầy đủ", "vừa đủ", "quá nhiều", "quá ít", "phải chăng", ]
pronoun_related_words = ["tôi", "tao", "mày", "cậu", "chúng tôi", "chúng ta", "họ", "bạn", "các bạn"]
intro_related_words = ["giới thiệu", "tìm kiếm", "tìm", "tôi muốn tìm", "tôi muốn giới thiệu", "tôi muốn tìm kiếm", "tôi muốn tìm hiểu về", "tôi muốn biết về", "hãy", "gợi ý"]
misc_words = ['giúp', 'cho', 'nhờ', 'về', 'có', 'là', 'được', 'cũng', 'nên', 'cần', 'phải', 'muốn', 'thích', 'yêu thích', 'ưa thích', 'chỗ']

limit_related_words = ["giới hạn", "hạn chế", "tối đa", "tối thiểu", "ít nhất", "nhiều nhất", "không quá", "không dưới"]

service_related_words = ["phục vụ", "đón tiếp", "hỗ trợ", "tư vấn", "giao hàng", "đặt bàn", "đặt món"]
puncuation_pattern = r'[^\w\s]'

purpose_related_words = ["ăn uống", "đi chơi", "hẹn hò", "gặp gỡ bạn bè", "tổ chức sự kiện", "họp mặt gia đình", "làm việc", "học tập", "thư giãn", "giải trí", "ăn vặt", "ăn gia đình", "hẹn hò", "bbq - món nướng", "họp nhóm", "đãi tiệc", "ăn chay", "giao tận nơi", "uống bia - nhậu", "hát hò", "chụp hình", "quay phim", "tiếp khách", "ăn bình dân", "quán cóc", "buffet", "takeaway", "mang về", "tổ chức tiệc cưới", "ngẵm cảnh", "thư giản", "đọc sách", "học bài", "nghe nhạc", "du lịch", "ăn fastfood", "tiếp khách sang trọng", "dành cho nam giới", "dã ngoại", "tiệc ngoài trời", "xem phim hd", "tổ chức hội thảo", "tiệc tận nơi", "mua sắm", "ăn chơi ngày tết"]
synonym_of_purpose_related_words = {
    "ăn uống": ["ăn uống", "ăn", "uống"],
    "đi chơi": ["đi chơi", "chơi", "vui chơi"],
    "hẹn hò": ["hẹn hò", "hẹn hẹn", "hò hẹn"],
    "hát hò": ["hát hò", "hát", "hò hò"],
    "chụp hình": ["chụp hình", "chụp ảnh"],
    "quay phim": ["quay phim", "quay video"],
    "tiếp khách": ["tiếp khách", "tiếp đãi", "tiếp"],
    "ăn bình dân": ["ăn bình dân", "ăn dân dã", "ăn bình dân"],
    "quán cóc": ["quán cóc", "quán vỉa hè", "quán lề đường"],
    "buffet": ["buffet", "ăn buffet", "tiệc buffet"],
    "takeaway": ["takeaway", "mang đi", "mang về"],
    "tổ chức tiệc cưới": ["tổ chức tiệc cưới", "tổ chức đám cưới", "tổ chức lễ cưới"],
    "ngắm cảnh": ["ngắm cảnh", "ngắm", "ngắm nhìn"],
    "thư giãn": ["thư giãn", "nghỉ ngơi", "giải lao", "giải trí"],
    "đọc sách": ["đọc sách", "đọc", "đọc truyện"],
    "học bài": ["học bài", "học", "học tập"],
    "nghe nhạc": ["nghe nhạc", "nghe", "nghe nhạc"],
    "du lịch": ["du lịch", "tham quan", "khám phá", "ngoại du"],
    "ăn fastfood": ["ăn fastfood", "ăn nhanh", "ăn nhanh"],
    "tiếp khách sang trọng": ["tiếp khách sang trọng", "tiếp khách cao cấp", "tiếp khách sang"],
    "dành cho nam giới": ["dành cho nam giới", "cho nam giới", "dành cho phái mạnh"],
    "dã ngoại": ["dã ngoại", "ngoài trời", "thăm quan ngoài trời"],
    "tiệc ngoài trời": ["tiệc ngoài trời", "tiệc sân vườn", "tiệc ngoài"],
    "xem phim hd": ["xem phim hd", "xem phim", "phim hd", "phim"],
    "tổ chức hội thảo": ["tổ chức hội thảo", "tổ chức seminar", "tổ chức workshop"],
    "tiệc tận nơi": ["tiệc tận nơi", "tiệc tại nhà", "tiệc tại chỗ"],
    "gặp gỡ bạn bè": ["gặp gỡ bạn bè", "gặp gỡ", "gặp mặt", "gặp"],
    "tổ chức sự kiện": ["tổ chức sự kiện", "tổ chức event", "tổ chức buổi tiệc"],
    "họp mặt gia đình": ["họp mặt gia đình", "họp mặt", "gặp gỡ gia đình"],
    "ăn vặt": ["ăn vặt", "ăn nhẹ", "ăn uống nhẹ"],
    "ăn gia đình": ["ăn gia đình", "ăn cùng gia đình", "ăn tại nhà"],
    "bbq - món nướng": ["bbq - món nướng", "nướng", "bbq"],
    "họp nhóm": ["họp nhóm", "họp mặt nhóm", "gặp gỡ nhóm"],
    "đãi tiệc": ["đãi tiệc", "đón tiếp", "tiệc tùng"],
    "ăn chay": ["ăn chay", "chay", "thực phẩm chay"],
    "giao tận nơi": ["giao tận nơi", "giao hàng tận nơi", "giao thức ăn"],
    "uống bia - nhậu": ["uống bia - nhậu", "uống bia", "nhậu"]
}
currency_related_words = ["đồng", "nghìn", "triệu", "vnd", "vnđ","k"]

cuisine_related_words = ["món", "món ăn", "ẩm thực", "đặc sản", "món truyền thống"]



In [5]:
def preprocess_line(line: list[tuple[str,str,str]]):
    remove_set = set(pronoun_related_words + stopwords + intro_related_words + misc_words)
    # remove all stopwords in line
    for word in remove_set:           
        line = [w for w in line if w[0] != word and re.search(puncuation_pattern, w[0]) is None]
        
    return line
def get_texts(sentence: list[tuple[str,str,str]]):
    return [word[0] for word in sentence]

In [6]:
query_dataset = pd.read_csv(actual_query_dataset_path)
X = query_dataset[["query", "restaurant_id"]]
y = query_dataset['label']
with open(actual_query_text_path, 'r', encoding='utf-8') as f:
    queries = f.read().splitlines()
    
train_queries, test_queries = train_test_split(queries, test_size=0.2, random_state=42)
X_train = X[X['query'].isin(train_queries)]
X_test = X[X['query'].isin(test_queries)]
y_train = y[X['query'].isin(train_queries)]
y_test = y[X['query'].isin(test_queries)]


In [7]:
store_meta_keywords_file_name = os.path.join(base_path, "..", "..", "tmp", "store_infos.csv")
if os.path.exists(store_meta_keywords_file_name):
    store_infos = pd.read_csv(store_meta_keywords_file_name, encoding='utf-8')
    store_infos = store_infos['MetaKeywords'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
else:
    data = pd.read_csv(DATASET_PATH)
    store_infos = data['MetaKeywords'].apply(lambda x: x.lower() if isinstance(x, str) else x)
    store_infos = store_infos.apply(lambda x: chunk(x) if isinstance(x, str) else x)
    store_infos.to_csv(f'{store_meta_keywords_file_name}', index=False, encoding='utf-8')
    

In [8]:
preprocessed_store_infos_file_path = os.path.join(base_path, "..", "..", "tmp", "preprocessed_store_infos.csv")
if os.path.exists(preprocessed_store_infos_file_path):
    preprocessed_store_infos = pd.read_csv(preprocessed_store_infos_file_path, encoding='utf-8')
    preprocessed_store_infos = preprocessed_store_infos['MetaKeywords'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
else:
    store_infos.dropna(inplace=True)
    preprocessed_store_infos = store_infos.apply(preprocess_line).apply(get_texts)
    preprocessed_store_infos.to_csv(f'{preprocessed_store_infos_file_path}', index=False, encoding='utf-8')


In [9]:
model_path = os.path.join(base_path, "..", "..", "tmp", "word2vec.model")
word2vec_model = models.Word2Vec(preprocessed_store_infos, vector_size=200, min_count=2, workers=4, seed=42)
word2vec_model.save(model_path)

In [10]:
set_of_words = set()
for line in preprocessed_store_infos:
    set_of_words.update(set(list(line)))

In [11]:

def preprocess_for_word2vec(query: str, word2vec_model: models.Word2Vec):
    parsed_query:list[tuple[str,str,str]] = chunk(query.lower())
    parsed_query = preprocess_line(parsed_query) 
    new_query = []
    for word in parsed_query:
        if word in word2vec_model.wv.index_to_key:
            new_query.append(word)
    return get_texts(new_query)

In [ ]:
def find_location(text: str):
    for word in location_related_words:
        r = re.search(r'\b' + word + r'\b', text)
        if r:
            return r.start()
    return -1

def extract_locations(sentence: list[tuple[str,str,str]]):
    all_location = []
    for i, word in enumerate(sentence):
        r = find_location(word[0])
        if r != -1:
            # true_location = word[0] + f" {sentence[i + 1][0]}"
            all_location.append(i)
    return all_location

def extract_time(sentence: list[tuple[str,str,str]]):
    all_time = []
    for i, word in enumerate(sentence):
        for tow in time_of_week_related_words + time_of_day_related_words:
            r = re.search(r'\b' + tow + r'\b', word[0])
            if r:
                all_time.append(i)
        for taw in time_amount_related_words:
            r = re.search(r'\b' + taw + r'\b', word[0])
            if r and i >= 2:
                all_time.append(i)
    return all_time

def extract_service(sentence: list[tuple[str,str,str]]):
    all_service = []
    for i, word in enumerate(sentence):
        for service in service_related_words:
            
            r = re.search(r'\b' + service + r'\b', word[0])
            if r:
                all_service.append(word[0])
    return all_service


data = pd.read_csv(actual_dataset_path)
def remove_empty(x):
    while '' in x:
        x.remove('')
    return x

cuisine_values = data["Cuisines"].str.lower().str.split('|')
cuisine_values.dropna(inplace=True)
cuisine_values = cuisine_values.apply(remove_empty)

cuisine_related_words = sorted(set(cuisine_related_words).union({
    cuisine.strip()
    for cuisines in cuisine_values
    for cuisine in cuisines
    if cuisine.strip()
}))

t = data["LstTargetAudience"].str.lower().str.split('|')
t.dropna(inplace=True)
t.apply(remove_empty)
dictionary = corpora.Dictionary(t)

dictionary.save(os.path.join(base_path, "..", "..", "tmp", "targetaudience_corpus.dict"))

def extract_for_who(sentence: list[tuple[str,str,str]]):
    all_target_score = []
    for i, word in enumerate(sentence):
        if word[0] in dictionary.token2id.keys():
            all_target_score.append(i)
    return all_target_score


def extract_cuisine(sentence: list[tuple[str, str, str]]):
    all_result = []
    for i, word in enumerate(sentence):
        for cuisine in cuisine_related_words:
            r = re.search(r'\b' + cuisine + r'\b', word[0])
            if r:
                all_result.append(i)
    return all_result

def to_minutes(time_point: str):
    if time_point is None:
        return -1
    hour, minute = time_point.split(':')
    return int(hour) * 60 + int(minute)

def parse_time_point(time_range: str):
    # DB time parse
    time_range_obj = ast.literal_eval(time_range)
    return (to_minutes(time_range_obj[0][0]), to_minutes(time_range_obj[0][1]))



def parse_price_tokens(sentence: list[str]):
    price_values = []
    seen_prices = set()
    multiplier_units = {"k", "nghìn", "nghin"}

    for i, token in enumerate(sentence):
        cleaned_token = token.strip().lower()
        if not cleaned_token:
            continue

        price_value = None
        if cleaned_token.endswith('k') and cleaned_token[:-1].isdigit():
            price_value = int(cleaned_token[:-1]) * 1000
        elif cleaned_token.isdigit():
            price_value = int(cleaned_token)
            if i + 1 < len(sentence):
                next_token = sentence[i + 1].strip().lower()
                if next_token in multiplier_units:
                    price_value *= 1000
        elif re.fullmatch(r'\d{1,3}(?:\.\d{3})+', cleaned_token):
            price_value = int(cleaned_token.replace('.', ''))

        if price_value is not None and price_value not in seen_prices:
            seen_prices.add(price_value)
            price_values.append(price_value)

    return price_values
    

In [23]:
def retrieve_time_in_minutes(tokens: list[str]) -> int:
    if not isinstance(tokens, list) or len(tokens) == 0:
        return -1

    minute_units = {"p", "phut", "phút"}
    hour_units = {"h", "gio", "giờ"}

    cleaned_tokens = [str(token).strip().lower() for token in tokens if str(token).strip()]

    for i, token in enumerate(cleaned_tokens):
        # Pattern: HH:MM
        m = re.fullmatch(r"(\d{1,2}):(\d{1,2})", token)
        if m:
            hour = int(m.group(1))
            minute = int(m.group(2))
            return hour * 60 + minute

        # Pattern: 10h30 or 10h
        m = re.fullmatch(r"(\d{1,2})h(\d{1,2})?", token)
        if m:
            hour = int(m.group(1))
            minute = int(m.group(2)) if m.group(2) else 0
            return hour * 60 + minute

        # Pattern: 10 giờ 30 phút or 10 h 30 p
        if token.isdigit() and i + 1 < len(cleaned_tokens) and cleaned_tokens[i + 1] in hour_units:
            hour = int(token)
            minute = 0
            if i + 3 < len(cleaned_tokens) and cleaned_tokens[i + 2].isdigit() and cleaned_tokens[i + 3] in minute_units:
                minute = int(cleaned_tokens[i + 2])
            return hour * 60 + minute

        # Pattern: 30 phút
        if token.isdigit() and i + 1 < len(cleaned_tokens) and cleaned_tokens[i + 1] in minute_units:
            return int(token)

    return -1

In [13]:
if not os.path.exists(os.path.join(base_path, "..", "..", "tmp", "attribute model")):
    os.makedirs(os.path.join(base_path, "..", "..", "tmp", "attribute model"), exist_ok=True)

cuisine_dictionary = corpora.Dictionary([cuisine_related_words])
cuisine_model_bm25 = models.OkapiBM25Model(dictionary=cuisine_dictionary)
cuisine_model_bm25.save(os.path.join(base_path, "..", "..", "tmp", "attribute model", "cuisine_bm25.model"))

t_model_bm25 = models.OkapiBM25Model(dictionary.doc2bow(text) for text in t)
t_model_bm25.save(os.path.join(base_path, "..", "..", "tmp", "attribute model", "targetaudience_bm25.model"))

service_dictionary = corpora.Dictionary([service_related_words])
service_model_bm25 = models.OkapiBM25Model(dictionary=service_dictionary)
service_model_bm25.save(os.path.join(base_path, "..", "..", "tmp", "attribute model", "service_bm25.model"))

def time_prepare(sentence: list[tuple[str, str, str]], locations):
    text = get_texts(sentence)
    if len(locations) == 0:
        return []
    selected_day = []
    selected_time = []
    for p in locations:
        if text[p] in time_amount_related_words and p >= 2 and \
           time.strptime(text[p-2] + ":" + text[p -1], '%H:%M'):
            t = to_minutes(text[p-2] + ":" + text[p -1])
            selected_time.append(t)
        elif text[p] in time_of_week_related_words:
            selected_day.append(text[p])
    
    selected_values = {}
    for day, sel_time in zip(selected_day, selected_time):
        selected_values[day] = sel_time
    return selected_values

def embedding(sentence: list[tuple[str, str, str]], locations, model, dictionary):
    all_result = []
    for p in locations:
        word = sentence[p]
        all_result.append(model[dictionary.doc2bow([word[0]])])
    return all_result


### Tọa độ thập phân (Decimal Degrees)

| Khu vực | Vĩ độ (lat) | Kinh độ (lon) |
|---|---:|---:|
| Quận 1 | 10.776111 | 106.695833 |
| Quận 2 | 10.778611 | 106.757500 |
| Quận 3 | 10.780000 | 106.679444 |
| Quận 4 | 10.761667 | 106.702500 |
| Quận 5 | 10.756667 | 106.666667 |
| Quận 6 | 10.746111 | 106.636111 |
| Quận 7 | 10.738611 | 106.726389 |
| Quận 8 | 10.723333 | 106.627778 |
| Quận 9 | 10.840000 | 106.770833 |
| Quận 10 | 10.773611 | 106.667222 |
| Quận 11 | 10.766944 | 106.645556 |
| Quận 12 | 10.861944 | 106.658889 |
| Bình Thạnh | 10.802778 | 106.696667 |
| Tân Bình | 10.803611 | 106.650833 |
| Phú Nhuận | 10.801667 | 106.677500 |
| Tân Phú | 10.792222 | 106.625278 |
| Gò Vấp | 10.841667 | 106.666667 |
| Bình Tân | 10.771111 | 106.590556 |
| Thủ Đức | 10.849722 | 106.768333 |
| Bình Chánh | 10.750278 | 106.512500 |
| Nhà Bè | 10.651667 | 106.726389 |
| Hóc Môn | 10.878333 | 106.592500 |
| Củ Chi | 11.027778 | 106.483056 |
| Cần Giờ | 10.511944 | 106.880556 |

In [14]:
# Tọa độ thập phân theo khu vực (key là tên khu vực)
geo_decimal = {
    "Quận 1": (10.776111, 106.695833),
    "Quận 2": (10.778611, 106.757500),
    "Quận 3": (10.780000, 106.679444),
    "Quận 4": (10.761667, 106.702500),
    "Quận 5": (10.756667, 106.666667),
    "Quận 6": (10.746111, 106.636111),
    "Quận 7": (10.738611, 106.726389),
    "Quận 8": (10.723333, 106.627778),
    "Quận 9": (10.840000, 106.770833),
    "Quận 10": (10.773611, 106.667222),
    "Quận 11": (10.766944, 106.645556),
    "Quận 12": (10.861944, 106.658889),
    "Bình Thạnh": (10.802778, 106.696667),
    "Tân Bình": (10.803611, 106.650833),
    "Phú Nhuận": (10.801667, 106.677500),
    "Tân Phú": (10.792222, 106.625278),
    "Gò Vấp": (10.841667, 106.666667),
    "Bình Tân": (10.771111, 106.590556),
    "Thủ Đức": (10.849722, 106.768333),
    "Bình Chánh": (10.750278, 106.512500),
    "Nhà Bè": (10.651667, 106.726389),
    "Hóc Môn": (10.878333, 106.592500),
    "Củ Chi": (11.027778, 106.483056),
    "Cần Giờ": (10.511944, 106.880556),
    "center": (10.0039, 106.0044)
}

geo_decimal

{'Quận 1': (10.776111, 106.695833),
 'Quận 2': (10.778611, 106.7575),
 'Quận 3': (10.78, 106.679444),
 'Quận 4': (10.761667, 106.7025),
 'Quận 5': (10.756667, 106.666667),
 'Quận 6': (10.746111, 106.636111),
 'Quận 7': (10.738611, 106.726389),
 'Quận 8': (10.723333, 106.627778),
 'Quận 9': (10.84, 106.770833),
 'Quận 10': (10.773611, 106.667222),
 'Quận 11': (10.766944, 106.645556),
 'Quận 12': (10.861944, 106.658889),
 'Bình Thạnh': (10.802778, 106.696667),
 'Tân Bình': (10.803611, 106.650833),
 'Phú Nhuận': (10.801667, 106.6775),
 'Tân Phú': (10.792222, 106.625278),
 'Gò Vấp': (10.841667, 106.666667),
 'Bình Tân': (10.771111, 106.590556),
 'Thủ Đức': (10.849722, 106.768333),
 'Bình Chánh': (10.750278, 106.5125),
 'Nhà Bè': (10.651667, 106.726389),
 'Hóc Môn': (10.878333, 106.5925),
 'Củ Chi': (11.027778, 106.483056),
 'Cần Giờ': (10.511944, 106.880556),
 'center': (10.0039, 106.0044)}

In [15]:
vector_coridinates = {}
for key in geo_decimal:
    v = (geo_decimal[key][0] - geo_decimal["center"][0], geo_decimal[key][1] - geo_decimal["center"][1])
    manitude = (v[0]**2 + v[1]**2)**0.5
    if manitude == 0:
        vector_coridinates[key] = 0, 0
    else:
        vector_coridinates[key] = v[0] / manitude, v[1] / manitude
vector_coridinates

{'Quận 1': (0.7449980257653432, 0.6670666695359174),
 'Quận 2': (0.7170371291660058, 0.6970349742999794),
 'Quận 3': (0.7545215450151054, 0.656275276166959),
 'Quận 4': (0.7354693073807269, 0.6775580402451985),
 'Quận 5': (0.750796591415108, 0.6605334800897343),
 'Quận 6': (0.7615180878660177, 0.648143658345034),
 'Quận 7': (0.7132550543247266, 0.7009045780134636),
 'Quận 8': (0.7557569792919335, 0.6548521880940248),
 'Quận 9': (0.7371506268841126, 0.6757284612063931),
 'Quận 10': (0.7577613517856151, 0.6525317875322528),
 'Quận 11': (0.765606500372923, 0.6433091687413801),
 'Quận 12': (0.7951005092412601, 0.6064776831873445),
 'Bình Thạnh': (0.7557330601602844, 0.6548797918555528),
 'Tân Bình': (0.7776974662614158, 0.6286387285003795),
 'Phú Nhuận': (0.7642989942673554, 0.6448620374637579),
 'Tân Phú': (0.7856002629119874, 0.6187343750856388),
 'Gò Vấp': (0.7844854810231408, 0.6201471841941167),
 'Bình Tân': (0.7946250179273244, 0.6071005525314561),
 'Thủ Đức': (0.7421180303639201, 0.

In [16]:
def compute_cosine_similarity(vec1, vec2):
    dot_product = vec1[0] * vec2[0] + vec1[1] * vec2[1]
    magnitude_vec1 = (vec1[0]**2 + vec1[1]**2)**0.5
    magnitude_vec2 = (vec2[0]**2 + vec2[1]**2)**0.5
    if magnitude_vec1 == 0 or magnitude_vec2 == 0:
        return 0.0
    return dot_product / (magnitude_vec1 * magnitude_vec2)

def compare_two_time(time1, time2):
    return abs(time1 - time2) / (24 * 3600)

def compare_time_ranges(range1, range2):
    start_diff = compare_two_time(range1[0], range2[0])
    end_diff = compare_two_time(range1[1], range2[1])
    return (start_diff + end_diff) / 2

def compare_price(price1, price2):
    if price1 == 0 or price2 == 0:
        return 0.0
    return abs(price1 - price2) / max(price1, price2)



In [22]:
parsed_data = data.copy()
metakeyword_preprocess = []
print(type(preprocessed_store_infos))
for v in preprocessed_store_infos:
    metakeyword_preprocess.append(word2vec_model.wv.get_vector(v) if v in word2vec_model.wv.index_to_key else [0] * word2vec_model.vector_size)
parsed_data['MetaKeywords'] = metakeyword_preprocess

<class 'pandas.core.series.Series'>


ValueError: Length of values (103594) does not match length of index (107365)

In [18]:
parsed_data['Cuisines'] = cuisine_values
cuisine_embeddings = []
for values in cuisine_values:
    cuisine_embeddings.append(cuisine_model_bm25[cuisine_dictionary.doc2bow(values)])
parsed_data['CuisineEmbeddings'] = cuisine_embeddings


ValueError: Length of values (91204) does not match length of index (107365)

In [ ]:
districts_embeddings = []
for district in parsed_data['District']:
    if district in vector_coridinates:
        districts_embeddings.append(vector_coridinates[district])
    else:
        districts_embeddings.append((0, 0))
parsed_data['DistrictEmbeddings'] = districts_embeddings


In [ ]:

for label in ["Thứ hai", "Thứ ba", "Thứ tư", "Thứ năm", "Thứ sáu", "Thứ bảy", "Chủ nhật"]:
    time_embeddings = []
    for t in parsed_data[label]:
        
        if pd.isna(t):
            time_embeddings.append((0, 0))
        else:
            
            time_range = parse_time_point(t)
            time_embeddings.append(time_range)
    parsed_data[label + "_embeddings"] = time_embeddings
    

## Linear Model Scaffold: Predict Rating from Query + Store Info

Mục tiêu: dự đoán `label` (rating 0-4) từ:
- Câu query trong `dataset/restaurant_dataset_ver1.csv`
- Thông tin quán trong `dataset/foody_combined_data_final.csv`

Luồng chính:
1. Đọc 2 dataset
2. Link bằng `restaurant_id`
3. Tạo text feature cho query + store
4. Train linear model (`Ridge`)
5. Đánh giá và tạo hàm dự đoán

In [30]:
import numpy as np
from pathlib import Path
from typing import Iterable

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Optional debug print settings
pd.set_option("display.max_columns", 120)

In [31]:
# Resolve project root from current notebook location
PROJECT_ROOT = Path.cwd().resolve().parents[1]

QUERY_PATH = PROJECT_ROOT / "dataset" / "restaurant_dataset_ver1.csv"
STORE_PATH = PROJECT_ROOT / "dataset" / "foody_combined_data_final.csv"

query_df = pd.read_csv(QUERY_PATH)
store_df = pd.read_csv(STORE_PATH)

print(f"query_df shape: {query_df.shape}")
print(f"store_df shape: {store_df.shape}")
print("query columns:", list(query_df.columns))
print("store columns:", list(store_df.columns)[:20], "...")

query_df shape: (7500, 8)
store_df shape: (107365, 37)
query columns: ['query', 'restaurant_name', 'restaurant_id', 'label', 'reason', 'district', 'cuisines', 'source']
store columns: ['RestaurantID', 'Name', 'Address', 'District', 'Area', 'PriceMin', 'PriceMax', 'MetaKeywords', 'Cuisines', 'LstTargetAudience', 'LstCategory', 'AccessGuide', 'RestaurantUrl', 'Vị trí', 'Giá cả', 'Chất lượng', 'Phục vụ', 'Không gian', 'Excellent', 'Good'] ...


In [32]:
def pick_id_col(df: pd.DataFrame, candidates: Iterable[str]) -> str:
    lower_map = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    raise ValueError(f"Cannot find id column in {list(df.columns)}")


def clean_text(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


# Adjust candidate list if your store table uses a different id field name
query_id_col = pick_id_col(query_df, ["restaurant_id"])
store_id_col = pick_id_col(store_df, ["restaurant_id", "id", "resid", "restaurantid"])

query_df[query_id_col] = pd.to_numeric(query_df[query_id_col], errors="coerce")
store_df[store_id_col] = pd.to_numeric(store_df[store_id_col], errors="coerce")

merged = query_df.merge(
    store_df,
    left_on=query_id_col,
    right_on=store_id_col,
    how="inner",
    suffixes=("_query", "_store"),
)

print(f"Merged rows: {len(merged)}")
print(f"Unique queries: {merged['query'].nunique() if 'query' in merged.columns else 'N/A'}")

Merged rows: 7500
Unique queries: 500


In [33]:
# Build compact text fields for model features.
# Candidate list is aligned with foody_combined_data_final.csv columns.
store_text_candidates = [
    "Name",
    "Address",
    "District",
    "Area",
    "MetaKeywords",
    "Cuisines",
    "LstTargetAudience",
    "LstCategory",
    "AccessGuide",
    "PriceMin",
    "PriceMax",
    "Vị trí",
    "Giá cả",
    "Chất lượng",
    "Phục vụ",
    "Không gian",
    "Excellent",
    "Good",
    "Average",
    "Bad",
    "HasBooking",
    "HasDelivery",
    "HasPromotion",
    "TotalView",
    "TotalFavourite",
    "TotalCheckins",
    "Giao tận nơi",
    "Đặt bàn",
]

available_store_cols = [c for c in store_text_candidates if c in merged.columns]
missing_store_cols = [c for c in store_text_candidates if c not in merged.columns]

if not available_store_cols:
    raise ValueError("No store feature columns found in merged dataframe.")

print("Available store feature columns:", available_store_cols)
if missing_store_cols:
    print("Missing store feature columns (safe to ignore):", missing_store_cols)

merged["query_text"] = clean_text(merged["query"])
merged["store_text"] = clean_text(
    merged[available_store_cols].fillna("").astype(str).agg(" ".join, axis=1)
)

# Keep only rows with valid label
target_col = "label"
model_df = merged.dropna(subset=[target_col]).copy()
model_df[target_col] = pd.to_numeric(model_df[target_col], errors="coerce")
model_df = model_df.dropna(subset=[target_col])

X = model_df[["query_text", "store_text"]]
y = model_df[target_col].astype(float)

print("Trainable rows:", len(model_df))
print("Label distribution:")
print(y.value_counts().sort_index())

Available store feature columns: ['Name', 'Address', 'District', 'Area', 'MetaKeywords', 'Cuisines', 'LstTargetAudience', 'LstCategory', 'AccessGuide', 'PriceMin', 'PriceMax', 'Vị trí', 'Giá cả', 'Chất lượng', 'Phục vụ', 'Không gian', 'Excellent', 'Good', 'Average', 'Bad', 'HasBooking', 'HasDelivery', 'HasPromotion', 'TotalView', 'TotalFavourite', 'TotalCheckins', 'Giao tận nơi', 'Đặt bàn']
Trainable rows: 7500
Label distribution:
label
0.0    3908
1.0    1602
2.0     675
3.0     594
4.0     721
Name: count, dtype: int64


In [34]:
from sklearn.model_selection import GroupShuffleSplit

# Split by query so the same user query never appears in both train and test.
groups = model_df["query"].astype(str)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(f"Train rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Train queries: {model_df.iloc[train_idx]['query'].nunique()}")
print(f"Test queries: {model_df.iloc[test_idx]['query'].nunique()}")

text_preprocessor = ColumnTransformer(
    transformers=[
        ("query_tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2)), "query_text"),
        ("store_tfidf", TfidfVectorizer(max_features=40000, ngram_range=(1, 2)), "store_text"),
    ],
    remainder="drop",
)

linear_model = Pipeline(
    steps=[
        ("fill_missing", FunctionTransformer(lambda df: df.fillna(""), validate=False)),
        ("features", text_preprocessor),
        ("regressor", Ridge(alpha=1.0, random_state=42)),
    ]
)

linear_model.fit(X_train, y_train)

# Change this value to evaluate the top-K cutoff used by NDCG, MRR, and HIT.
TOP_K = 10
relevance_threshold = 3

# Ranking metrics are computed per query group on the test set.
def dcg_at_k(relevances: list[float], k: int) -> float:
    scores = np.asarray(relevances[:k], dtype=float)
    if scores.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, scores.size + 2))
    gains = np.power(2.0, scores) - 1.0
    return float(np.sum(gains / discounts))


def ndcg_at_k(relevances: list[float], k: int) -> float:
    actual = dcg_at_k(relevances, k)
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)
    return 0.0 if ideal == 0 else actual / ideal


ranking_rows = model_df.iloc[test_idx].copy()
ranking_rows["score"] = linear_model.predict(X_test)
ranking_rows["is_relevant"] = ranking_rows["label"].astype(float) >= relevance_threshold

query_metrics = []
for query, group in ranking_rows.groupby("query", sort=False):
    ordered = group.sort_values("score", ascending=False)
    relevances = ordered["label"].astype(float).tolist()
    binary_relevance = ordered["is_relevant"].astype(bool).tolist()

    ndcg_k = ndcg_at_k(relevances, TOP_K)

    hit_k = 1.0 if any(binary_relevance[:TOP_K]) else 0.0
    mrr_k = 0.0
    for rank, is_relevant in enumerate(binary_relevance[:TOP_K], start=1):
        if is_relevant:
            mrr_k = 1.0 / rank
            break

    query_metrics.append(
        {
            "query": query,
            "ndcg": ndcg_k,
            "mrr": mrr_k,
            "hit": hit_k,
        }
    )

metrics_df = pd.DataFrame(query_metrics)
print(f"Evaluated queries: {len(metrics_df)}")
print(f"NDCG@{TOP_K}: {metrics_df['ndcg'].mean():.4f}")
print(f"MRR@{TOP_K}: {metrics_df['mrr'].mean():.4f}")
print(f"HIT@{TOP_K}: {metrics_df['hit'].mean():.4f}")
print(metrics_df.head())

Train rows: 6000
Test rows: 1500
Train queries: 400
Test queries: 100
Evaluated queries: 100
NDCG@10: 0.6280
MRR@10: 0.4993
HIT@10: 0.8200
                                               query      ndcg       mrr  hit
0  Tôi muốn tìm 1 quán ăn hải sản ở quận 6 cho tố...  0.333333  0.142857  1.0
1  Hãy tìm kiếm giúp tôi một nhà hàng Ý ở quận 1 ...  0.817190  1.000000  1.0
2  Tôi muốn tìm một nhà hàng sang trọng món miền ...  0.526821  0.000000  0.0
3       Tìm chỗ ăn chill, đồ ăn ổn, phục vụ không tệ  0.574347  0.333333  1.0
4  Ở quận 1 có quán món Việt nào hợp tiếp khách v...  0.719147  1.000000  1.0


In [29]:
# Build lookup table for store features by restaurant_id (for inference)
store_feature_table = (
    merged[[query_id_col, "store_text"]]
    .drop_duplicates(subset=[query_id_col])
    .set_index(query_id_col)
)


def predict_rating(user_query: str, restaurant_id: int) -> dict:
    """Predict rating for a (query, restaurant_id) pair.

    Returns:
        dict: Contains continuous prediction, clipped prediction, and rounded 0-4 rating.
    """
    if restaurant_id not in store_feature_table.index:
        return {
            "ok": False,
            "message": f"restaurant_id={restaurant_id} not found in linked dataset",
        }

    store_text = store_feature_table.loc[restaurant_id, "store_text"]
    input_df = pd.DataFrame(
        {
            "query_text": [str(user_query).strip().lower()],
            "store_text": [store_text],
        }
    )

    pred = float(linear_model.predict(input_df)[0])
    pred_clip = float(np.clip(pred, 0, 4))
    pred_round = int(np.rint(pred_clip))

    return {
        "ok": True,
        "prediction_continuous": pred,
        "prediction_clipped": pred_clip,
        "prediction_rounded_0_4": pred_round,
    }


# Example usage
sample = predict_rating(
    "Tôi muốn tìm quán nướng BBQ ở Tân Phú phù hợp gia đình cuối tuần",
    656449,
)
sample

{'ok': True,
 'prediction_continuous': 1.542548121385564,
 'prediction_clipped': 1.542548121385564,
 'prediction_rounded_0_4': 2}

In [30]:
import re
import ast
import math
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from gensim.models import Word2Vec


# ------------------------
# Global config and state
# ------------------------
DATASET_FILENAME = "foody_combined_data_final.csv"
STOPWORDS_FILENAME = "stopwords.txt"
META_PREPROCESSED_FILENAME = "preprocessed_store_infos.csv"
W2V_VECTOR_SIZE = 200
TIME_MARGIN_MINUTES = 180

DAY_COLS = ["Thứ hai", "Thứ ba", "Thứ tư", "Thứ năm", "Thứ sáu", "Thứ bảy", "Chủ nhật"]
DAY_ALIASES = {
    0: ["thu hai", "thu 2"],
    1: ["thu ba", "thu 3"],
    2: ["thu tu", "thu 4"],
    3: ["thu nam", "thu 5"],
    4: ["thu sau", "thu 6"],
    5: ["thu bay", "thu 7"],
    6: ["chu nhat"],
}
TIME_SLOT_DEFAULTS = {
    "sang": (7, 0, 11, 0),
    "trua": (11, 0, 13, 30),
    "chieu": (14, 0, 17, 30),
    "toi": (18, 0, 22, 30),
    "dem": (18, 0, 22, 30),
}
LOCATION_PREFIX_RE = r"^(?:quan|q|huyen|h|thi xa|tx|thanh pho|tp|phuong|p|xa)\s+"

SERVICE_SYNONYMS = {
    "giao_tan_noi": ["giao tận nơi", "giao hang", "giao hàng", "ship"],
    "dat_ban": ["đặt bàn", "dat ban", "booking", "đặt chỗ", "dat cho"],
    "khuyen_mai": ["khuyến mãi", "khuyen mai", "ưu đãi", "uu dai", "promotion"],
    "phuc_vu": ["phục vụ", "phuc vu", "nhân viên", "nhan vien", "service"],
}
SERVICE_VOCAB = {key: i for i, key in enumerate(SERVICE_SYNONYMS.keys())}

ASPECT_ALIAS = {
    "vi_tri": ["vi tri", "dia diem", "dia chi"],
    "gia_ca": ["gia ca", "gia", "re", "dat", "phai chang"],
    "chat_luong": ["chat luong", "ngon", "do an ngon"],
    "phuc_vu": ["phuc vu", "nhan vien", "service"],
    "khong_gian": ["khong gian", "view", "dep", "thoang"],
}
RATING_ORDER = ["vi_tri", "gia_ca", "chat_luong", "phuc_vu", "khong_gian"]

# runtime globals
_STORE_DF_FOR_PARSE: pd.DataFrame = pd.DataFrame()
_STOPWORDS_RAW: set[str] = set()
_STOPWORDS_CANON: set[str] = set()
_LOCATION_VOCAB: dict[str, int] = {}
_LOCATION_LABELS: dict[str, str] = {}
_LOCATION_ALIAS_TO_KEY: dict[str, str] = {}
_CUISINE_VOCAB: dict[str, int] = {}
_CUISINE_LABELS: dict[str, str] = {}
_TARGET_VOCAB: dict[str, int] = {}
_TARGET_LABELS: dict[str, str] = {}
_CATEGORY_VOCAB: dict[str, int] = {}
_CATEGORY_LABELS: dict[str, str] = {}
_META_CORPUS: list[list[str]] = []
_META_W2V: Word2Vec | None = None


def _normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).lower()).strip()


def _strip_accents(text: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", str(text)) if unicodedata.category(c) != "Mn")


def _canon(text: str) -> str:
    return _strip_accents(_normalize_text(text))


def _query_tokens(user_query: str, drop_stopwords: bool = True) -> list[str]:
    text = re.sub(r"[^\w\s]", " ", _canon(user_query))
    tokens = [t for t in text.split() if t]
    if drop_stopwords:
        tokens = [t for t in tokens if t not in _STOPWORDS_CANON]
    return tokens


def _query_text(user_query: str, drop_stopwords: bool = True) -> str:
    return " ".join(_query_tokens(user_query, drop_stopwords=drop_stopwords))


def _split_pipe_values(raw_value) -> list[str]:
    if pd.isna(raw_value):
        return []
    return [x.strip() for x in re.findall(r"[^|]+", str(raw_value)) if x.strip()]


def _is_positive(raw_value) -> bool:
    if pd.isna(raw_value):
        return False
    if isinstance(raw_value, (int, float, np.integer, np.floating)):
        return float(raw_value) > 0
    text = _canon(raw_value)
    return text in {"1", "true", "yes", "co", "c"} or "co" in text


def _to_minutes(hour: int, minute: int) -> int:
    hour = max(0, min(23, int(hour)))
    minute = max(0, min(59, int(minute)))
    return hour * 60 + minute


def _minutes_to_cyc(minutes: int) -> np.ndarray:
    rad = 2 * math.pi * (float(minutes) / 1440.0)
    return np.array([math.sin(rad), math.cos(rad)], dtype=float)


def _find_existing_path(filename: str, include_model_folder: bool = False) -> Path | None:
    roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    candidates = []
    for root in roots:
        candidates.append(root / filename)
        candidates.append(root / "dataset" / filename)
        candidates.append(root / "tmp" / filename)
        if include_model_folder:
            candidates.append(root / "model" / "linear model" / filename)

    for p in candidates:
        rp = p.resolve()
        if rp.exists():
            return rp
    return None


def _load_store_df() -> pd.DataFrame:
    if "store_df" in globals() and isinstance(store_df, pd.DataFrame):
        return store_df.copy()
    path = _find_existing_path(DATASET_FILENAME)
    if path is None:
        raise FileNotFoundError(f"Cannot locate {DATASET_FILENAME}")
    return pd.read_csv(path)


def _load_stopwords() -> set[str]:
    if "stopwords" in globals() and isinstance(stopwords, list):
        return {str(x).strip() for x in stopwords if str(x).strip()}

    path = _find_existing_path(STOPWORDS_FILENAME, include_model_folder=True)
    if path is None:
        return set()

    with open(path, "r", encoding="utf-8") as f:
        return {x.strip() for x in f.read().split() if x.strip()}


def _load_meta_corpus() -> list[list[str]]:
    if "preprocessed_store_infos" in globals():
        corpus = []
        for row in preprocessed_store_infos:
            if isinstance(row, list):
                tokens = [str(t).strip().lower() for t in row if str(t).strip()]
            elif isinstance(row, str):
                try:
                    parsed = ast.literal_eval(row)
                    if isinstance(parsed, list):
                        tokens = [str(t).strip().lower() for t in parsed if str(t).strip()]
                    else:
                        tokens = re.findall(r"[\wÀ-ỹ]+", _normalize_text(row))
                except Exception:
                    tokens = re.findall(r"[\wÀ-ỹ]+", _normalize_text(row))
            else:
                tokens = []
            if tokens:
                corpus.append(tokens)
        if corpus:
            return corpus

    path = _find_existing_path(META_PREPROCESSED_FILENAME)
    if path is None:
        return []

    df = pd.read_csv(path, encoding="utf-8")
    col = "MetaKeywords" if "MetaKeywords" in df.columns else df.columns[0]
    corpus = []
    for raw in df[col].dropna().astype(str):
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                tokens = [str(t).strip().lower() for t in parsed if str(t).strip()]
            else:
                tokens = re.findall(r"[\wÀ-ỹ]+", _normalize_text(raw))
        except Exception:
            tokens = re.findall(r"[\wÀ-ỹ]+", _normalize_text(raw))
        if tokens:
            corpus.append(tokens)
    return corpus


def _build_vocab(terms: list[str]) -> tuple[dict[str, int], dict[str, str]]:
    canon_to_raw = {}
    for term in terms:
        c = _canon(term)
        if c and c not in canon_to_raw:
            canon_to_raw[c] = str(term).strip()
    vocab = {c: i for i, c in enumerate(sorted(canon_to_raw.keys()))}
    return vocab, canon_to_raw


def _build_location_alias_map(location_vocab: dict[str, int]) -> dict[str, str]:
    alias_to_key = {}
    for key in sorted(location_vocab.keys(), key=len, reverse=True):
        alias_to_key.setdefault(key, key)
        stripped = re.sub(LOCATION_PREFIX_RE, "", key).strip()
        if stripped:
            alias_to_key.setdefault(stripped, key)
            alias_to_key.setdefault(f"quan {stripped}", key)
            alias_to_key.setdefault(f"huyen {stripped}", key)
    return alias_to_key


def _vector_from_terms(terms: list[str], vocab: dict[str, int]) -> np.ndarray:
    vec = np.zeros(len(vocab), dtype=float)
    for term in terms:
        c = _canon(term)
        if c in vocab:
            vec[vocab[c]] = 1.0
    return vec


def _extract_vocab_matches(user_query: str, vocab_keys: list[str], max_hits: int = 6) -> list[str]:
    query = _query_text(user_query, drop_stopwords=True)
    matches = []
    for key in sorted(vocab_keys, key=len, reverse=True):
        if key and re.search(r"(?<!\w)" + re.escape(key) + r"(?!\w)", query):
            matches.append(key)
            if len(matches) >= max_hits:
                break
    return matches


def _extract_location_matches(user_query: str, max_hits: int = 3) -> list[str]:
    query = _query_text(user_query, drop_stopwords=False)
    hits = []
    seen = set()
    for alias in sorted(_LOCATION_ALIAS_TO_KEY.keys(), key=len, reverse=True):
        if re.search(r"(?<!\w)" + re.escape(alias) + r"(?!\w)", query):
            key = _LOCATION_ALIAS_TO_KEY[alias]
            if key not in seen:
                seen.add(key)
                hits.append(key)
                if len(hits) >= max_hits:
                    break
    return hits


def _price_token_to_vnd(number_text: str, unit_text: str | None) -> int:
    value = float(str(number_text).replace(".", "").replace(",", "."))
    unit = _canon(unit_text or "")
    if unit in {"k", "nghin", "ngan"}:
        value *= 1000
    elif unit in {"tr", "trieu"}:
        value *= 1_000_000
    return int(value)


def _parse_open_close_from_text(raw_value) -> tuple[int, int] | None:
    if pd.isna(raw_value):
        return None
    text = _canon(raw_value)

    hhmm = re.findall(r"(\d{1,2}):(\d{1,2})", text)
    if len(hhmm) >= 2:
        return (_to_minutes(int(hhmm[0][0]), int(hhmm[0][1])), _to_minutes(int(hhmm[1][0]), int(hhmm[1][1])))

    compact = re.findall(r"(\d{1,2})h(\d{0,2})", text)
    if len(compact) >= 2:
        m1 = int(compact[0][1]) if compact[0][1] else 0
        m2 = int(compact[1][1]) if compact[1][1] else 0
        return (_to_minutes(int(compact[0][0]), m1), _to_minutes(int(compact[1][0]), m2))

    return None


def _collect_store_schedules(store_row: pd.Series) -> dict[int, tuple[int, int]]:
    schedules = {}
    for idx, day_col in enumerate(DAY_COLS):
        if day_col in store_row.index:
            open_close = _parse_open_close_from_text(store_row[day_col])
            if open_close is not None:
                schedules[idx] = open_close
    return schedules


def _embed_meta_tokens(tokens: list[str]) -> np.ndarray:
    if _META_W2V is None or not tokens:
        return np.zeros(W2V_VECTOR_SIZE, dtype=float)
    vecs = []
    for token in tokens:
        t = _normalize_text(token)
        if t in _META_W2V.wv:
            vecs.append(_META_W2V.wv[t])
    if not vecs:
        return np.zeros(W2V_VECTOR_SIZE, dtype=float)
    return np.mean(np.asarray(vecs, dtype=float), axis=0)


def _init_runtime() -> None:
    global _STORE_DF_FOR_PARSE
    global _STOPWORDS_RAW, _STOPWORDS_CANON
    global _LOCATION_VOCAB, _LOCATION_LABELS, _LOCATION_ALIAS_TO_KEY
    global _CUISINE_VOCAB, _CUISINE_LABELS
    global _TARGET_VOCAB, _TARGET_LABELS
    global _CATEGORY_VOCAB, _CATEGORY_LABELS
    global _META_CORPUS, _META_W2V

    _STORE_DF_FOR_PARSE = _load_store_df()

    _STOPWORDS_RAW = _load_stopwords()
    _STOPWORDS_CANON = {_canon(x) for x in _STOPWORDS_RAW if _canon(x)}

    location_terms = []
    for col in ["District", "Area"]:
        if col in _STORE_DF_FOR_PARSE.columns:
            location_terms.extend(_STORE_DF_FOR_PARSE[col].dropna().astype(str).tolist())

    cuisine_terms = []
    if "Cuisines" in _STORE_DF_FOR_PARSE.columns:
        for raw in _STORE_DF_FOR_PARSE["Cuisines"].dropna().astype(str):
            cuisine_terms.extend(_split_pipe_values(raw))

    target_terms = []
    if "LstTargetAudience" in _STORE_DF_FOR_PARSE.columns:
        for raw in _STORE_DF_FOR_PARSE["LstTargetAudience"].dropna().astype(str):
            target_terms.extend(_split_pipe_values(raw))

    category_terms = []
    if "LstCategory" in _STORE_DF_FOR_PARSE.columns:
        for raw in _STORE_DF_FOR_PARSE["LstCategory"].dropna().astype(str):
            category_terms.extend(_split_pipe_values(raw))

    _LOCATION_VOCAB, _LOCATION_LABELS = _build_vocab(location_terms)
    _LOCATION_ALIAS_TO_KEY = _build_location_alias_map(_LOCATION_VOCAB)
    _CUISINE_VOCAB, _CUISINE_LABELS = _build_vocab(cuisine_terms)
    _TARGET_VOCAB, _TARGET_LABELS = _build_vocab(target_terms)
    _CATEGORY_VOCAB, _CATEGORY_LABELS = _build_vocab(category_terms)

    _META_CORPUS = _load_meta_corpus() or [["empty"]]
    if "word2vec_model" in globals() and hasattr(word2vec_model, "wv") and int(getattr(word2vec_model, "vector_size", 0)) == W2V_VECTOR_SIZE:
        _META_W2V = word2vec_model
    else:
        _META_W2V = Word2Vec(
            sentences=_META_CORPUS,
            vector_size=W2V_VECTOR_SIZE,
            window=5,
            min_count=1,
            workers=2,
            seed=42,
        )


_init_runtime()


def compare_embeddings(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    if vec_a.shape != vec_b.shape:
        return 0.0
    denom = float(np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    if denom == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / denom)


def build_store_location_embedding(store_row: pd.Series) -> np.ndarray:
    terms = [str(store_row[col]) for col in ["District", "Area"] if col in store_row.index and pd.notna(store_row[col])]
    return _vector_from_terms(terms, _LOCATION_VOCAB)


def build_store_cuisine_embedding(store_row: pd.Series) -> np.ndarray:
    terms = _split_pipe_values(store_row["Cuisines"]) if "Cuisines" in store_row.index and pd.notna(store_row["Cuisines"]) else []
    return _vector_from_terms(terms, _CUISINE_VOCAB)


def build_store_target_audience_embedding(store_row: pd.Series) -> np.ndarray:
    terms = _split_pipe_values(store_row["LstTargetAudience"]) if "LstTargetAudience" in store_row.index and pd.notna(store_row["LstTargetAudience"]) else []
    return _vector_from_terms(terms, _TARGET_VOCAB)


def build_store_category_embedding(store_row: pd.Series) -> np.ndarray:
    terms = _split_pipe_values(store_row["LstCategory"]) if "LstCategory" in store_row.index and pd.notna(store_row["LstCategory"]) else []
    return _vector_from_terms(terms, _CATEGORY_VOCAB)


def build_store_service_embedding(store_row: pd.Series) -> np.ndarray:
    vec = np.zeros(len(SERVICE_VOCAB), dtype=float)
    if _is_positive(store_row.get("HasDelivery", np.nan)) or _is_positive(store_row.get("Giao tận nơi", np.nan)):
        vec[SERVICE_VOCAB["giao_tan_noi"]] = 1.0
    if _is_positive(store_row.get("HasBooking", np.nan)) or _is_positive(store_row.get("Đặt bàn", np.nan)):
        vec[SERVICE_VOCAB["dat_ban"]] = 1.0
    if _is_positive(store_row.get("HasPromotion", np.nan)):
        vec[SERVICE_VOCAB["khuyen_mai"]] = 1.0

    service_text = " ".join(
        str(store_row[c])
        for c in ["AccessGuide", "MetaKeywords"]
        if c in store_row.index and pd.notna(store_row[c])
    )
    service_canon = _canon(service_text)
    for key, aliases in SERVICE_SYNONYMS.items():
        if any(_canon(alias) in service_canon for alias in aliases):
            vec[SERVICE_VOCAB[key]] = 1.0
    return vec


def build_store_rating_embedding(store_row: pd.Series) -> np.ndarray:
    aspect_cols = {
        "vi_tri": ["Vị trí", "Vị Trí"],
        "gia_ca": ["Giá cả", "Giá Cả"],
        "chat_luong": ["Chất lượng", "Chất Lượng"],
        "phuc_vu": ["Phục vụ", "Phục Vụ"],
        "khong_gian": ["Không gian", "Không Gian"],
    }

    def _pick_float(candidates: list[str]) -> float:
        for c in candidates:
            if c in store_row.index and pd.notna(store_row[c]):
                try:
                    return float(store_row[c])
                except Exception:
                    pass
        return np.nan

    vals = [_pick_float(aspect_cols[k]) for k in RATING_ORDER]
    if np.isnan(vals).all():
        exc = float(store_row.get("Excellent", 0) or 0)
        good = float(store_row.get("Good", 0) or 0)
        avg = float(store_row.get("Average", 0) or 0)
        bad = float(store_row.get("Bad", 0) or 0)
        total = exc + good + avg + bad
        general = 0.0 if total == 0 else 10.0 * (exc + 0.75 * good + 0.5 * avg + 0.25 * bad) / total
        vals = [general] * 5

    arr = np.array([0.0 if np.isnan(v) else float(v) for v in vals], dtype=float)
    return np.concatenate([arr, np.array([float(arr.mean())], dtype=float)])


def build_store_time_embedding(store_row: pd.Series, preferred_day_index: int | None = None) -> np.ndarray:
    schedules = _collect_store_schedules(store_row)
    if preferred_day_index is not None and preferred_day_index in schedules:
        day_idx = preferred_day_index
    elif schedules:
        day_idx = sorted(schedules.keys())[0]
    else:
        day_idx = 0

    open_min, close_min = schedules.get(day_idx, (0, 0))
    day_vec = np.zeros(7, dtype=float)
    day_vec[day_idx] = 1.0
    return np.concatenate([_minutes_to_cyc(open_min), _minutes_to_cyc(close_min), day_vec])


def _day_distance(day_a: int, day_b: int) -> int:
    diff = abs(int(day_a) - int(day_b))
    return min(diff, 7 - diff)


def _directional_margin_score(delta_good: int, cap: int = TIME_MARGIN_MINUTES) -> float:
    cap = max(1, int(cap))
    if delta_good >= 0:
        return min(1.0, 0.7 + 0.3 * (delta_good / cap))
    return max(0.0, 0.7 + (delta_good / cap))


def compare_store_time_fit(user_time: dict, store_row: pd.Series, margin_minutes: int = TIME_MARGIN_MINUTES) -> float:
    """Return [0,1] fitness for how well a store satisfies requested time.

    Exact open/close at requested time gets baseline 0.7.
    Earlier opening and later closing get higher scores than exact matching.
    """
    schedules = _collect_store_schedules(store_row)
    if not schedules:
        return 0.0

    req_open = int(user_time.get("opening_minute", 0))
    req_close = int(user_time.get("closing_minute", req_open))
    req_day = user_time.get("day_index", None)

    candidates = []
    if req_day is None:
        candidates = [(d, oc[0], oc[1], 1.0) for d, oc in schedules.items()]
    elif req_day in schedules:
        oc = schedules[req_day]
        candidates = [(req_day, oc[0], oc[1], 1.0)]
    else:
        for d, oc in schedules.items():
            day_weight = max(0.4, 1.0 - (_day_distance(req_day, d) / 3.5))
            candidates.append((d, oc[0], oc[1], day_weight))

    best = 0.0
    for _, open_min, close_min, day_weight in candidates:
        open_fit = _directional_margin_score(req_open - open_min, margin_minutes)
        close_fit = _directional_margin_score(close_min - req_close, margin_minutes)
        coverage_bonus = 1.0 if (open_min <= req_open and close_min >= req_close) else 0.85
        score = ((open_fit + close_fit) / 2.0) * day_weight * coverage_bonus
        if score > best:
            best = score

    return float(np.clip(best, 0.0, 1.0))


def compare_time_from_query(user_query: str, store_row: pd.Series, margin_minutes: int = TIME_MARGIN_MINUTES) -> float:
    return compare_store_time_fit(parse_user_time(user_query), store_row, margin_minutes=margin_minutes)


def parse_user_location(user_query: str) -> dict:
    hits = _extract_location_matches(user_query, max_hits=3)
    return {
        "value": [_LOCATION_LABELS[h] for h in hits if h in _LOCATION_LABELS],
        "embedding": _vector_from_terms(hits, _LOCATION_VOCAB),
    }


def parse_user_price(user_query: str) -> tuple[float, float]:
    text_canon = _query_text(user_query, drop_stopwords=False)
    pattern = r"(\d+(?:[\.,]\d+)?)\s*(k|nghin|ngan|trieu|tr|vnd|d|dong)?"
    values = []

    for match in re.finditer(pattern, text_canon):
        num, unit = match.group(1), match.group(2)
        start, end = match.span()
        left_context = text_canon[max(0, start - 12):start]
        local_context = text_canon[max(0, start - 12):min(len(text_canon), end + 12)]

        if re.search(r"(quan|thu)\s*$", left_context):
            continue

        try:
            value_vnd = _price_token_to_vnd(num, unit)
        except Exception:
            continue

        if unit is None:
            if value_vnd < 1000:
                continue
            if not re.search(r"gia|price|duoi|tren|toi da|toi thieu|khoang|tam|tu", local_context):
                continue

        values.append(value_vnd)

    if not values:
        return (0.0, float("inf"))
    if len(values) >= 2:
        return (float(min(values[0], values[1])), float(max(values[0], values[1])))

    single = float(values[0])
    if re.search(r"(tu|tren|it nhat|toi thieu)", text_canon):
        return (single, float("inf"))
    return (0.0, single)


def parse_user_cuisine(user_query: str) -> dict:
    hits = _extract_vocab_matches(user_query, list(_CUISINE_VOCAB.keys()), max_hits=6)
    return {
        "value": [_CUISINE_LABELS[h] for h in hits if h in _CUISINE_LABELS],
        "embedding": _vector_from_terms(hits, _CUISINE_VOCAB),
    }


def parse_user_target_audience(user_query: str) -> dict:
    hits = _extract_vocab_matches(user_query, list(_TARGET_VOCAB.keys()), max_hits=6)
    return {
        "value": [_TARGET_LABELS[h] for h in hits if h in _TARGET_LABELS],
        "embedding": _vector_from_terms(hits, _TARGET_VOCAB),
    }


def parse_user_categories(user_query: str) -> dict:
    hits = _extract_vocab_matches(user_query, list(_CATEGORY_VOCAB.keys()), max_hits=6)
    return {
        "value": [_CATEGORY_LABELS[h] for h in hits if h in _CATEGORY_LABELS],
        "embedding": _vector_from_terms(hits, _CATEGORY_VOCAB),
    }


def parse_user_rating(user_query: str) -> dict:
    text = _query_text(user_query, drop_stopwords=False)
    requested = [aspect for aspect, aliases in ASPECT_ALIAS.items() if any(a in text for a in aliases)]

    desired_score = None
    for pattern in [
        r"(\d+(?:[\.,]\d+)?)\s*/\s*10",
        r"(\d+(?:[\.,]\d+)?)\s*(?:diem|sao|star)",
        r"danh gia\s*(\d+(?:[\.,]\d+)?)",
    ]:
        m = re.search(pattern, text)
        if m:
            score = float(m.group(1).replace(",", "."))
            if 0 <= score <= 10:
                desired_score = score
                break

    if desired_score is None:
        if any(w in text for w in ["excellent", "xuat sac", "rat tot", "tuyet voi"]):
            desired_score = 9.5
        elif any(w in text for w in ["tot", "dep", "on", "ok"]):
            desired_score = 8.0
        elif any(w in text for w in ["trung binh", "average"]):
            desired_score = 5.0
        elif any(w in text for w in ["te", "xau", "bad", "kem"]):
            desired_score = 2.0
        else:
            desired_score = 7.0

    aspect_vec = np.zeros(5, dtype=float)
    if requested:
        for i, aspect in enumerate(RATING_ORDER):
            aspect_vec[i] = desired_score if aspect in requested else 0.0
    else:
        aspect_vec[:] = desired_score

    emb = np.concatenate([aspect_vec, np.array([desired_score], dtype=float)])
    return {"requested_aspects": requested, "desired_score": desired_score, "embedding": emb}


def parse_user_time(user_query: str) -> dict:
    text = _query_text(user_query, drop_stopwords=False)

    day_index = None
    for idx, aliases in DAY_ALIASES.items():
        if any(a in text for a in aliases):
            day_index = idx
            break

    open_min = None
    close_min = None

    am_pm = re.search(r"(\d{1,2})(?::(\d{1,2}))?\s*(am|pm)", text)
    if am_pm:
        hour = int(am_pm.group(1))
        minute = int(am_pm.group(2)) if am_pm.group(2) else 0
        if am_pm.group(3) == "pm" and hour < 12:
            hour += 12
        if am_pm.group(3) == "am" and hour == 12:
            hour = 0
        open_min = _to_minutes(hour, minute)
        close_min = open_min

    if open_min is None:
        range_match = re.search(r"(\d{1,2})(?:[:hg](\d{1,2}))?\s*(?:-|den|toi|to)\s*(\d{1,2})(?:[:hg](\d{1,2}))?", text)
        if range_match:
            h1 = int(range_match.group(1))
            m1 = int(range_match.group(2)) if range_match.group(2) else 0
            h2 = int(range_match.group(3))
            m2 = int(range_match.group(4)) if range_match.group(4) else 0
            open_min = _to_minutes(h1, m1)
            close_min = _to_minutes(h2, m2)
        else:
            single_match = re.search(r"(\d{1,2})(?:[:hg](\d{1,2}))?", text)
            if single_match:
                h = int(single_match.group(1))
                m = int(single_match.group(2)) if single_match.group(2) else 0
                open_min = _to_minutes(h, m)
                close_min = open_min

    if any(x in text for x in ["chieu", "toi", "dem"]):
        if open_min is not None and open_min < 12 * 60:
            open_min += 12 * 60
        if close_min is not None and close_min < 12 * 60:
            close_min += 12 * 60
    elif "sang" in text:
        if open_min == 12 * 60:
            open_min = 0
        if close_min == 12 * 60:
            close_min = 0

    if open_min is None or close_min is None:
        slot = next((name for name in TIME_SLOT_DEFAULTS if name in text), None)
        if slot is None:
            open_min, close_min = _to_minutes(0, 0), _to_minutes(23, 59)
        else:
            h1, m1, h2, m2 = TIME_SLOT_DEFAULTS[slot]
            open_min, close_min = _to_minutes(h1, m1), _to_minutes(h2, m2)

    if day_index is None:
        day_vec = np.full(7, 1.0 / 7.0, dtype=float)
    else:
        day_vec = np.zeros(7, dtype=float)
        day_vec[day_index] = 1.0

    emb = np.concatenate([_minutes_to_cyc(open_min), _minutes_to_cyc(close_min), day_vec])
    return {"day_index": day_index, "opening_minute": int(open_min), "closing_minute": int(close_min), "embedding": emb}


def parse_user_service(user_query: str) -> dict:
    text = _query_text(user_query, drop_stopwords=True)
    labels = [key for key, aliases in SERVICE_SYNONYMS.items() if any(_canon(alias) in text for alias in aliases)]
    vec = np.zeros(len(SERVICE_VOCAB), dtype=float)
    for key in labels:
        vec[SERVICE_VOCAB[key]] = 1.0
    return {"value": labels, "embedding": vec}


def parse_user_misc(user_query: str) -> dict:
    tokens = _query_tokens(user_query, drop_stopwords=True)
    return {"value": tokens, "embedding": _embed_meta_tokens(tokens)}


def parse_user_all(user_query: str) -> dict:
    return {
        "location": parse_user_location(user_query),
        "price": parse_user_price(user_query),
        "cuisine": parse_user_cuisine(user_query),
        "target_audience": parse_user_target_audience(user_query),
        "categories": parse_user_categories(user_query),
        "rating": parse_user_rating(user_query),
        "time": parse_user_time(user_query),
        "service": parse_user_service(user_query),
        "misc": parse_user_misc(user_query),
    }

In [31]:
# Quick validation for time-fit scoring behavior and location extraction
user_time = parse_user_time("quán mở lúc 3 pm thứ 6")

score_open_2pm = compare_store_time_fit(user_time, pd.Series({"Thứ sáu": "14:00 - 21:00"}))
score_open_3pm = compare_store_time_fit(user_time, pd.Series({"Thứ sáu": "15:00 - 21:00"}))

score_close_15 = compare_store_time_fit(user_time, pd.Series({"Thứ sáu": "10:00 - 15:00"}))
score_close_17 = compare_store_time_fit(user_time, pd.Series({"Thứ sáu": "10:00 - 17:00"}))

location_check = parse_user_location("Tôi muốn tìm quán ăn ở khu vực Tân Bình")

print("open@2pm > open@3pm:", score_open_2pm, ">", score_open_3pm)
print("close@5pm > close@3pm:", score_close_17, ">", score_close_15)
print("location parse:", location_check["value"])

open@2pm > open@3pm: 0.8999999999999999 > 0.85
close@5pm > close@3pm: 0.95 > 0.85
location parse: ['Quận Tân Bình']


In [ ]:
# Parsing the whole dataset and building embeddings for the first 5 stores as a sanity check
